# **Crowd Behavior Analysis Using CNNs and Aerial Video Signals**
The project processes a self-made drone video dataset to classify crowd behaviors using a CNN-LSTM model.

### ***1. Importing libraries***

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, TimeDistributed, GlobalAveragePooling2D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### **2. Global Configuration & Dataset Path**

In [ ]:

VIDEO_FOLDER = r"C:\Users\hp\Desktop\ANN Project\crowd_behavior_data" 
SEQUENCE_LENGTH = 16
IMG_SIZE = (112, 112)  
CLASSES = ['normal', 'panic', 'aggressive', 'idle']


### **3. Function to Extract Sequences from Video**

In [ ]:
def extract_sequences_from_video(video_path, sequence_length=SEQUENCE_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, IMG_SIZE)
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    cap.release()
    sequences = []

    for i in range(0, len(frames) - sequence_length + 1, sequence_length):
        sequence = frames[i:i + sequence_length]
        sequences.append(np.array(sequence, dtype=np.float32))

    return sequences



### **4. Data Loading, Preprocessing & Sequence Generation**

In [ ]:
data = []
labels = []

datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

for class_name in CLASSES:
    class_dir = os.path.join(VIDEO_FOLDER, class_name)
    if not os.path.exists(class_dir):
        continue

    image_files = sorted(
        [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.png'))],
        key=lambda x: int(''.join(filter(str.isdigit, x)) or 0)
    )

    for i in range(len(image_files) - SEQUENCE_LENGTH + 1):
        sequence = []
        for j in range(SEQUENCE_LENGTH):
            img_path = os.path.join(class_dir, image_files[i + j])
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0
            img = datagen.random_transform(img)
            sequence.append(img)

        if len(sequence) == SEQUENCE_LENGTH:
            data.append(sequence)
            labels.append(CLASSES.index(class_name))

data = np.array(data, dtype=np.float32)
labels = to_categorical(np.array(labels), num_classes=len(CLASSES))
print(f"✅ Loaded {len(data)} sequences. Shape: {data.shape}")

 ### **5. Data Splitting: Train, Validation & Test**

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42) 

print(f"✅ Data split:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")



### **6. CNN-LSTM Hybrid Model Definition**


In [ ]:
def build_cnn_lstm_model():
    cnn_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    cnn_base.trainable = False

    model = Sequential([
        TimeDistributed(cnn_base, input_shape=(SEQUENCE_LENGTH, IMG_SIZE[0], IMG_SIZE[1], 3)),
        TimeDistributed(GlobalAveragePooling2D()),
        LSTM(64, return_sequences=False),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(len(CLASSES), activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_cnn_lstm_model()
model.summary()

#### Only cnn model

In [ ]:
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import TimeDistributed, GlobalAveragePooling2D, Dense, Dropout, Flatten
#from tensorflow.keras.applications import MobileNetV2
#from tensorflow.keras.optimizers import Adam
#
#def build_cnn_model():
#    # Load pre-trained MobileNetV2 without the top layer
#    cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
#    cnn.trainable = False
#
#    model = Sequential([
#        TimeDistributed(cnn, input_shape=(SEQUENCE_LENGTH, IMG_SIZE[0], IMG_SIZE[1], 3)),
#        TimeDistributed(GlobalAveragePooling2D()),
#        Flatten(),  # Flatten the sequence output
#        Dense(128, activation='relu'),
#        Dropout(0.5),
#        Dense(len(CLASSES), activation='softmax')
#    ])
#
#    model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
#    return model
#
#model = build_cnn_model()
#model.summary()
#


 ### **7. Training Function**

In [ ]:

#def train_model(model, X_train, y_train, X_val, y_val):
#    model_path = 'best_model.h5'
#    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#    model_checkpoint = ModelCheckpoint(model_path, save_best_only=True, monitor='val_loss', mode='min', verbose=1)
#
#    history = model.fit(
#        X_train, y_train,
#        validation_data=(X_val, y_val),
#        epochs=20,
#        batch_size=8,
#        callbacks=[early_stopping, model_checkpoint],
#        verbose=1
#    )
#    return history
#
#history = train_model(model, X_train, y_train, X_val, y_val)


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

def train_or_load_model(model, X_train, y_train, X_val, y_val, model_path='best_model.h5'):
    abs_model_path = os.path.abspath(model_path)

    # Check if model already exists
    if os.path.exists(model_path):
        print(f"📦 Loading saved model from: {abs_model_path}")
        loaded_model = load_model(model_path)
        return loaded_model, None  # No new training
    else:
        print("🚀 No saved model found. Starting training...")
        print(f"📝 Model will be saved to: {abs_model_path}")
        
        # Callbacks
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
        model_checkpoint = ModelCheckpoint(model_path, save_best_only=True, monitor='val_loss', mode='min', verbose=1)
        
        # Train model
        history = model.fit(
            X_train, y_train, 
            epochs=10,
            batch_size=4,
            validation_data=(X_val, y_val),
            callbacks=[early_stopping, model_checkpoint],
            verbose=1
        )

        # Check if file saved
        if os.path.exists(model_path):
            print(f"✅ Model successfully saved at: {abs_model_path}")
        else:
            print("❌ Model was not saved. Please check checkpoint configuration.")

        return model, history

# Usage example (assumes 'model', 'X_train', etc. are already defined):
model, history = train_or_load_model(model, X_train, y_train, X_val, y_val)



### **8. Model Evaluation on Test Set**

In [ ]:

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print("Classification Report:\n")
print(classification_report(y_true_classes, y_pred_classes, target_names=CLASSES))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# **Testing Part for external Input**

### **Loading Already Extracted Frames from Folder (for Prediction)**

In [ ]:

def load_frames_from_folder(frame_dir, img_size=(224, 224)):
    frame_files = sorted([
        f for f in os.listdir(frame_dir) if f.lower().endswith(('.jpg', '.png'))
    ])
    
    frames = []
    for frame_file in frame_files:
        frame_path = os.path.join(frame_dir, frame_file)
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        frame = cv2.resize(frame, img_size)
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    print(f"✅ Loaded {len(frames)} frames from '{frame_dir}'")
    return frames

frames = load_frames_from_folder(r"C:\Users\hp\Desktop\ANN Project\test2")




 ### **9. Extract frames from video for testing (Optional)**

In [ ]:

def extract_frames_from_video(video_path, save_dir='test_frames', img_size=(224,224)):
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        resized_frame = cv2.resize(frame, img_size)
        frame_path = os.path.join(save_dir, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_path, resized_frame)
        frame_count += 1

    cap.release()
    print(f"Total frames saved: {frame_count}")

# Use example: extract_frames_from_video(r"C:\Users\hp\Desktop\test video.mp4")



### **10. Combined Inference & Prediction Summary on Frames**

In [ ]:
def predict_behavior_on_frames(model_path, frame_dir='test2', sequence_length=SEQUENCE_LENGTH, img_size=IMG_SIZE, class_names=CLASSES):
    model = load_model(model_path)
    frame_files = sorted(os.listdir(frame_dir))
    frames = []
    all_predictions = []

    for i, frame_file in enumerate(frame_files):
        frame_path = os.path.join(frame_dir, frame_file)
        frame = cv2.imread(frame_path)
        frame = cv2.resize(frame, img_size)
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

        if len(frames) == sequence_length:
            input_seq = np.expand_dims(np.array(frames), axis=0)  # Shape: (1, seq_len, H, W, C)
            prediction = model.predict(input_seq)[0]
            class_index = np.argmax(prediction)
            class_label = class_names[class_index]

            print(f"Prediction at frame {i}: {class_label}")
            all_predictions.append(class_label)
            frames.pop(0)  # Slide window

    # Summary of predictions
    prediction_counts = Counter(all_predictions)
    print("\n📊 Prediction Summary:")
    for label, count in prediction_counts.items():
        print(f"  {label}: {count} times")

    most_common = prediction_counts.most_common(1)[0]
    print(f"\n🏆 Dominant Predicted Class: {most_common[0]} ({most_common[1]} frames)")

# --- Call the function here (outside the function) ---
predict_behavior_on_frames('best_model.h5', frame_dir='test2')
